In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Database Connection
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

complex_mapping = {
    'exter_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'exter_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'heating_qc': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'kitchen_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'fireplace_qu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'pool_qc': {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'None': 0},
    'lot_shape': {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1},
    'land_slope': {'Gtl': 3, 'Mod': 2, 'Sev': 1},
    'bsmt_exposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'garage_finish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0},
    'utilities': {'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1},
    'functional': {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
}

def ordinal_encode(df, mapping):
    df = df.copy()
    for col, m in mapping.items():
        if col in df.columns:
            df[col] = df[col].map(m).fillna(0).astype(int)
    return df

# 2. Load Model and Data
model = joblib.load('../models/ames_xgb_model.pkl')
X_test = pd.read_sql("SELECT * FROM X_test", engine)
y_test = pd.read_sql("SELECT * FROM y_test", engine).values.ravel()

# 3. Preprocess Test Set
X_test_encoded = ordinal_encode(X_test, complex_mapping)

# 4. Predictions 
log_predictions = model.predict(X_test_encoded)

# 5. REVERSE LOG TRANSFORMATION
# np.expm1 = (exp(x) - 1)
real_predictions = np.expm1(log_predictions)
real_y_test = np.expm1(y_test)

# 6. Metrics Calculation
r2 = r2_score(real_y_test, real_predictions)
mae = mean_absolute_error(real_y_test, real_predictions)

print(f"Final Test R2 Score: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:,.2f}")